# Aufgabe 3: Communication Patterns

Bevor wir uns die Vergleiche ansehen versuche ich kurz die Unterschiede grob zu erklären.

## **Vergleich der Kommunikationsmuster (Kafka vs. RabbitMQ)**

Beide Systeme agieren als Nachrichten-Broker und denoch basieren sie auf fundamental unterschiedlichen Paradigmen:

### **Architektur-Paradigma:**

- **Kafka (Event Streaming Platform):** Kafka nutzt ein **Log-basiertes** Paradigma. Man nennt es auch "Dumb Broker / Smart Consumer". Nachrichten werden als unverändliche Events sequenziell auf die Festplatte geschrieben (in Partitionen). Der Broker merkt sich nicht welche Nachricht gelesen wurde, stattdessen verwaltet der Konsument seinen eigenen Lese-Fortschritt (Offset).

- **RabbitMQ (Message Broker)**: RabbitMQ nutzt ein tradtionelles **Queue-basiertes** Paradigma. Es ist sogesagt der Gegenteil von Kafka "Smart Broker / Dumb Consumer". Der Broker merkt sich den Status jeder Nachricht. Wenn ein Konsument eine Nachricht liest und bestätigt (Ack), wird diese standardmässig gelöscht.

### **Routing**:

- **Kafka**: Ziemlich rudimentär. Man sendet an ein Topic und Nachrichten werden anhand eines Keys auf Partitionen verteilt.

- **RabbitMQ**: Sehr mächtig. Es nutzt das AMQP-Protokoll<sup>*</sup> mit exchanges und Bindings. Nachrichten können nach exakten Mustern (Direct), Themen oder an alle verteilt werden, bevor sie in der Queue landen.

<small>* Advanced Message Queuing Protocol, man kann sich das wie ein Paketpost vorstellen: <br> Sender (Produzent) -> Postamt (Broker/RabbitMQ) -> Empfänger (Konsument)</small>

### **Vorteile / Nachteile**:

- Kafka Vorteil: Persistent und "Time-Travel". Mann kann alte Events jederzeit neu abspielen (wie in Deep Dive gezeigt bei Aufgabe 2)

- RabbitMQ Vorteil: Flexibles Routing und exzellente Unterstützung für das klassische "Worker-Queue"-Muster (Competing Consumers).

- Kafka Nachteil: Komplexer im Setup und in der Skalierung (limitiert durch die Anzahl der Partitionen).

- RabbitMQ Nachteil: Nicht für die dauerhafte Speicherung historischer Daten gedacht.


### **Bekannte Probleme und deren Mitigation**

Beide Systeme haben unter hoher Last spezifische Schwachstellen:

- **RabbitMQ: Queue Buildup & OOM (Out of Memory)**

- Problem: Wenn die Data Sinks (Konsumenten) langsamer schreiben als die Generatoren produzieren, stauen sich die Nachrichten in der Queue. Da RabbitMQ versucht, Queues im RAM zu halten (für maximale Geschwindigkeit), kann dies den Arbeitsspeicher des Servers sprengen und den Broker zum Absturz bringen.

- Mitigation: Nutzung von "Lazy Queues" (die Nachrichten direkt auf die Disk schreiben) oder das Setzen einer TTL (Time-To-Live) mit einer "Dead Letter Exchange" (DLX), um unlesbare Nachrichten auszusortieren, bevor sie das System überlasten.

- **Kafka: Rebalancing Storms & Head-of-Line Blocking**

- Problem: Wenn in einer Consumer Group (wie bei der Skalierung im Deep Dive 2) ständig Instanzen hoch- und runterfahren, stoppt Kafka kurzzeitig die Verarbeitung, um die Partitionen neu zuzuweisen. Zudem blockiert eine fehlerhafte Nachricht in einer Partition alle nachfolgenden Nachrichten in derselben Partition.

- Mitigation: Erhöhung von Timeouts (session.timeout.ms), um unnötige Rebalances zu vermeiden, und Implementierung von isolierten "Dead Letter Topics" in der Applikationslogik für fehlerhafte Nachrichten.

### **Skalierbarkeit: Herausforderungen beider Ansätze**

- **Kafka (Skalierung durch Partitionen):**

- Die Skalierung ist vertikal an das Topic-Design gebunden. Wenn ein Topic 3 Partitionen hat, können maximal 3 Konsumenten in einer Consumer Group aktiv gleichzeitig Daten verarbeiten. Führt man einen 4. Konsumenten hinzu, bleibt dieser untätig (Idle). Um weiter zu skalieren, müssen die Partitionen im laufenden Betrieb erhöht werden, was den Message-Key-Hash (wie in Deep Dive 1 analysiert) brechen kann.


- **RabbitMQ (Skalierung durch Competing Consumers):**

- Hier ist die Skalierung der Konsumenten beinahe unbegrenzt. Man kann problemlos 50 Worker-Container starten, die alle aus derselben room_environment_mp-Queue lesen. RabbitMQ verteilt die Nachrichten per Round-Robin.

- Herausforderung: Der Flaschenhals ist die Queue selbst. Eine Standard-Queue lebt nur auf einem einzigen Broker-Knoten. Wenn dieser Knoten ausfällt, ist die Queue weg. Dies muss durch "Quorum Queues" mitigiert werden, die über mehrere Nodes repliziert werden, was wiederum massiv Netzwerkbandbreite kostet.

## Experiment: Skalierbarkeit in der Praxis

Um die theoretischen Unterschiede praktisch zu veranschaulichen, haben wir das System mit 5 Prozessoren getestet:

| Consumer | Kafka (3 Partitionen) | RabbitMQ (1 Queue) |
| :--- | :---: | :---: |
| **processor-1** | ✅ Active | ✅ Active |
| **processor-2** | ✅ Active | ✅ Active |
| **processor-3** | ✅ Active | ✅ Active |
| **processor-4** | ❌ Idle (wartet) | ✅ Active |
| **processor-5** | ❌ Idle (wartet) | ✅ Active |

<br>

```text
┌─────────────────────────────────────────────────────────────────────────────┐
│                        SCALABILITY COMPARISON                               │
├─────────────────────────────────────────────────────────────────────────────┤
│  KAFKA                                    RABBITMQ                          │
│  ══════                                   ═════════                         │
│  ┌─────────────┐                          ┌─────────────┐                   │
│  │   Topic     │                          │   Queue     │                   │
│  │ room_env_mp │                          │ room_env_mp │                   │
│  │ Part 0      │◄──── consumer-1          │             │◄──── consumer-1   │
│  │ Part 1      │◄──── consumer-2          │  Messages   │◄──── consumer-2   │
│  │ Part 2      │◄──── consumer-3          │  Round-     │◄──── consumer-3   │
│  │             │     (idle) consumer-4 ───┼─ Robin ─────│◄──── consumer-4   │
│  └─────────────┘                          │             │◄──── consumer-5   │
│       ▲                                   └─────────────┘                   │
│       │                                         ▲                           │
│   Producers                                  Producers                      │
│                                                                             │
│  MAX ACTIVE CONSUMERS: 3 (partitions)     MAX ACTIVE CONSUMERS: ALL (N)     │
└─────────────────────────────────────────────────────────────────────────────┘
```

**Beobachtung / Interpretation:**
- **Kafka**: Die Skalierung ist durch die Anzahl Partitionen begrenzt. Mit `NUM_PARTITIONS=3` können maximal 3 Konsumenten aktiv arbeiten. Consumer 4 & 5 bleiben im Leerlauf.
- **RabbitMQ**: Alle 5 Consumer empfangen Nachrichten im Round-Robin-Verfahren. Solange Nachrichten in der Queue sind, bleiben alle Consumer aktiv beschäftigt.

![RabbitMQ scalability](images/task3_scalability.png)